# Activity: Limits of Guess-And-Check Optimization

## Learning Objectives

By the end of this activity, you will be able to:

-   Describe the computational complexity of guess-and-check methods for
    optimizing predictive models.
-   Use an off-the-shelf optimization function to fit a predictive model
    more quickly than could be achieved with guess-and-check.

## Introduction

So far, we’ve studied the linear model $\hat{y} = f(x) = wx + b$ and
seen how to fit it to data by minimizing a measure of score. We’ve done
the minimization process using a guess-and-check approach, where we
sample a large number of values for the slope $w$ and intercept $b$,
compute the score for each pair, and select the pair with the lowest
score.

However, this approach has severe limitations when it comes to scaling
up to more complex models. In this activity, we’ll explore some of these
limitations.

Here is a function which fits a linear model with multiple input
features using a grid-search. In more detail, the model we are fitting
now has the form

$$
\begin{aligned}
    \hat{y} = f(x_1,x_2,\ldots,x_p) = w_1x_1 + w_2 x_2 + \cdots + w_px_p + b = \sum_{i = 1}^p w_i x_i + b\;.
\end{aligned}
$$

To do grid search, we are going to try out many, evenly-spaced values
for each weight $w_i$ and for the bias $b$. For example, we might try 20
evenly spaced values for each weight in the range $[-0.1, 0.1]$ and 20
evenly spaced values for the bias in the range $[0, 10]$. Then, we will
evaluate every possible combination of weights and bias from these sets.

## Problem 1

Suppose that I want to fit a linear model with just one input feature
($p = 1$) using this grid-search approach. If I want $k$ evenly-spaced
values for the weight and $k$ evenly-spaced values for the bias, how
many total combinations of weight and bias will I need to evaluate?

*[TODO: Your response here]*

## Problem 2

Suppose now that I want to fit a linear model with $p \geq 1$ input
features. How many combinations of weight and bias do I need to
evaluate?

*[TODO: Your response here]*

## Problem 3

### Intro

The function below implements the grid-search approach for fitting a
linear model with multiple input features. It accepts as input a matrix
of input features $X$ (with one row per data point and one column per
feature), a vector of output values $y$, a scoring function, and the
number of grid points to evaluate for each parameter. It returns the
best-fitting weights, bias, and score found via grid search.

In [ ]:
import numpy as np
from itertools import product
import numpy as np

def mse(y_true, y_pred):
    residuals = y_true - y_pred
    return np.mean(residuals ** 2)

def fit_linear_model(X, y, score_function = mse, n_grid = 20, **score_kwargs):

    # initialize estimates for the weights and bias
    best_w = None
    best_b = None
    best_score = float('inf')

    # set the parameter grids over which to search
    w_grid = np.linspace(-0.1, 0.1, n_grid)
    b_grid = np.linspace(0, 10, n_grid)
    
    # figure out how many features there are, needed 
    # to figure out how many weights we need
    num_features = X.shape[1]

    # possible weights: 
    possible_weights = product(w_grid, repeat=num_features)

    # main loop: try all combinations of weights and biases
    for b in b_grid:
        for w in possible_weights:
            
            w = np.array(w)

            # this is a fancy way to implement linear prediction without a for-loop

            # the main benefit here is that it is faster
            y_pred = X @ w + b

            # compute score and store updated best parameters if score is better
            # than any previous
            score = score_function(y, y_pred, **score_kwargs)

            if score < best_score:
                best_w = w
                best_b = b
                best_score = score

    return np.array(best_w), best_b, best_score

To try out this function, let’s download the abalone data set again:

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/middcs/data-science-notes/refs/heads/main/data/abalone/abalone.csv"
df = pd.read_csv(url)

We can extract a subset of the features and the target variable (the
number of rings):

In [ ]:
y = df["rings"].values
X = df[["length (mm)", "diameter (mm)", "height (mm)", "whole_weight (g)", "shucked_weight (g)", "viscera_weight (g)", "shell_weight (g)"]].values

Running the code below will run `fit_linear_model` on this data set with
just 3 grid points for each parameter. We’ve included a `%timeit` magic
command, which has the effect of repeating the function call multiple
times and reporting the average runtime.

In [ ]:
%timeit w, b, score = fit_linear_model(X, y, mse, n_grid = 3)

Now that we’ve done the timing experiment, we can print out the
best-fitting weights, bias, and score found:

In [ ]:
w, b, score = fit_linear_model(X, y, mse, n_grid = 3)
print("Best slopes:", w)
print("Best intercept:", b)
print("Best score:", score)

1.  How long did it take to run the code above with `n_grid = 3`?
2.  Give an estimate of how long it would take to run the same code with
    the same number of features, but `n_grid = 6`. What about
    `n_grid = 60`? Explain your reasoning.
3.  Give an estimate of how long it would take to run the same code with
    `n_grid = 3`, but with twice as many features. Explain your
    reasoning.
4.  In modern applications, we often need to fit models with hundreds or
    thousands of features. Based on your reasoning above, do you think
    that this grid-search approach is feasible for such applications?
    Why or why not?

*[TODO: Your response here]*

### Problem 4

The `scipy` library implements a `minimize` function which can be used
to algorithmically search for the combination of weights and bias which
minimize a given loss function. To make this work, we need to pass a
function called `loss` which accepts as input a vector of weights and a
bias combined (like $(w_1,w_2, \ldots,w_p, b)$), and returns the loss
(score) for those parameters. The code below shows how to do this for
the abalone data set using mean squared error as the loss function. It
also uses `%timeit` to measure the runtime of the optimization process.

In [ ]:
from scipy.optimize import minimize

def loss(w, b):
    return mse(y, X @ w + b)

initial_params = np.zeros(X.shape[1]+1)

%timeit result = minimize(lambda params: loss(params[:-1], params[-1]), initial_params)

result = minimize(lambda params: loss(params[:-1], params[-1]), initial_params)

result = minimize(lambda params: loss(params[:-1], params[-1]), initial_params)
optimized_w = result.x[:-1]
optimized_b = result.x[-1]  
optimized_score = loss(optimized_w, optimized_b)
print("Optimized slopes:", optimized_w)
print("Optimized intercept:", optimized_b)
print("Optimized score:", optimized_score)

Please answer the following questions:

1.  Approximately how fast is the `minimize` approach compared to the
    grid-search approach with `n_grid = 3`?
2.  Approximately how fast is the `minimize` approach compared to your
    estimate for grid search with `n_grid = 6`? What about
    `n_grid = 60`?
3.  Is the grid-search solution or the `minimize` solution better in
    terms of minimizing the score?

*[TODO: Your response here]*

***Note***: The `minimize` function uses techniques called *gradient
methods* to find the optimal weights and bias with accuracy which
typically exceeds any grid search. Gradient methods are beyond the scope
of this course, but you can learn about them more in courses like CSCI
0451: Machine Learning.

## Collaboration statement

In a markdown cell below, briefly list who or what you collaborated with
and how. Cite any sources here or with relevant inline comments in your
code. Acknowledge all contributors, both people and AI, and what
portions of this notebook they contributed. You do not need to cite or
acknowledge any material provided in the starter file(s).

## Submitting your notebook

You will simultaneously submit the following two files to the relevant
assignment on [Gradescope](https://gradescope.com) via the “Upload
option” (guide
[here](https://guides.gradescope.com/hc/en-us/articles/21865616724749-Submitting-a-Code-assignment)).
**Both files must be uploaded at the same time and the file names must
match the specification exactly for the autotesting to run
successfully.**

1.  `activity_need_for_optimization.ipynb`: Your completed IPython
    notebook. You can obtain this via the “File→Download→Download
    .ipynb” menu option in Colab.
2.  `activity_need_for_optimization.py`: Your completed IPython notebook
    as a Python file. You can obtain this via the
    “File→Download→Download .py” menu option in Colab. This file is used
    to provide line-level feedback on your submission.

You can submit multiple times, with only the most recent submission
(before the final due date) assessed for credit. Gradescope will run a
series of automated unit tests on your notebook (which may takes 10s of
seconds depending on the complexity of the notebook). Note that the
tests performed by Gradescope are limited. Passing all of the visible
tests does not guarantee that your submission correctly satisfies all of
the requirements of the assignment.